# Paper Figures (final — download here)

_Investigation `paper-figures` — coder reproduction notebook._

The publication figures for the meta-modeler's guide paper, each keeping the BioRender illustration and replacing the hand-drawn bigraph diagram with a bigraph-loom render.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/meta-modelers-guide/meta-modelers-guide').is_dir():
    REPO = Path('/home/runner/work/meta-modelers-guide/meta-modelers-guide')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from meta_modelers_guide.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Interfaces & composition (process bigraph) (`fig-02`)

**Question.** How does a process bigraph make shared state and dependencies explicit?

**Purpose.** A process bigraph: processes interacting with stores through ports and wires, making shared state and dependencies explicit.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `fig02-process-bigraph` | `meta_modelers_guide.composites.fig02-process-bigraph` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig02-process-bigraph`** — `spec_meta_modelers_guide_composites_fig02_process_bigraph` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig02_process_bigraph = load_spec(REPO / 'meta_modelers_guide/composites/fig02-process-bigraph.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig02_process_bigraph)

In [ ]:
# === Edit parameters for composite 'fig02-process-bigraph' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'p1'  (local:BigraphLink)
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p1']['config']['interval'] = 1.0
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p1']['config']['summary'] = 'Process p — connects place-graph nodes via typed ports'
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p1']['config']['contract']['status'] = 'draft - no update'
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p1']['config']['contract']['summary'] = 'Process p — connects place-graph nodes via typed ports'
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p1']['config']['contract']['description'] = 'A process in the process bigraph: it connects nodes of the place graph through its typed ports, replacing a Milner hyperedge in the link graph.'
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p1']['config']['contract']['ports']['in'] = 'a node this process reads'
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p1']['config']['contract']['ports']['out'] = 'a node this process writes'

# process 'p2'  (local:BigraphLink)
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p2']['config']['interval'] = 1.0
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p2']['config']['summary'] = 'Process p — connects place-graph nodes via typed ports'
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p2']['config']['contract']['status'] = 'draft - no update'
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p2']['config']['contract']['summary'] = 'Process p — connects place-graph nodes via typed ports'
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p2']['config']['contract']['description'] = 'A process in the process bigraph: it connects nodes of the place graph through its typed ports, replacing a Milner hyperedge in the link graph.'
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p2']['config']['contract']['ports']['in'] = 'a node this process reads'
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p2']['config']['contract']['ports']['out'] = 'a node this process writes'

# process 'p3'  (local:BigraphLink)
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p3']['config']['interval'] = 1.0
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p3']['config']['summary'] = 'Process p — connects place-graph nodes via typed ports'
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p3']['config']['contract']['status'] = 'draft - no update'
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p3']['config']['contract']['summary'] = 'Process p — connects place-graph nodes via typed ports'
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p3']['config']['contract']['description'] = 'A process in the process bigraph: it connects nodes of the place graph through its typed ports, replacing a Milner hyperedge in the link graph.'
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p3']['config']['contract']['ports']['in'] = 'a node this process reads'
spec_meta_modelers_guide_composites_fig02_process_bigraph['state']['p3']['config']['contract']['ports']['out'] = 'a node this process writes'

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-02 ===
STUDY = 'fig-02'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Figure 1**


In [ ]:
# Figure 1
show_viz(_render_one('svg:visualizations/figure_1.svg', {'caption': 'A process bigraph: processes interacting with stores through ports and wires, making shared state and dependencies explicit.'}, RUNS_DB, STUDY_YAML))

## Study: Orchestration (`fig-03`)

**Question.** How are processes orchestrated across timescales, workflows, and structural rewrites?

**Purpose.** Orchestration: (a) multi-timestepping coordinates processes updating at different rates through a shared store; (b) workflow organizes processes as a dependency DAG; (c) event-driven graph rewrites (division, engulfment, bursting) modify system structure.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `fig01b-multiscale-composite` | `meta_modelers_guide.composites.fig01b-multiscale-composite` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig01b-multiscale-composite`** — `spec_meta_modelers_guide_composites_fig01b_multiscale_composite` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig01b_multiscale_composite = load_spec(REPO / 'meta_modelers_guide/composites/fig01b-multiscale-composite.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig01b_multiscale_composite)

In [ ]:
# === Edit parameters for composite 'fig01b-multiscale-composite' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-03 ===
STUDY = 'fig-03'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Figure 2**


In [ ]:
# Figure 2
show_viz(_render_one('svg:visualizations/figure_2.svg', {'caption': 'Orchestration: (a) multi-timestepping coordinates processes updating at different rates through a shared store; (b) workflow organizes processes as a dependency DAG; (c) event-driven graph rewrites (division, engulfment, bursting) modify system structure.'}, RUNS_DB, STUDY_YAML))

## Study: The cellular interface (`fig-04`)

**Question.** What ports and contract define the minimal cellular interface?

**Purpose.** The minimal cellular interface: a cell exposes physical exchange ports (chemical, mechanical, electrical, thermal) and higher-level cellular ports (growth rate, shape, signaling, objective, viability).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `fig04b-cellular-interface` | `meta_modelers_guide.composites.fig04b-cellular-interface` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig04b-cellular-interface`** — `spec_meta_modelers_guide_composites_fig04b_cellular_interface` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig04b_cellular_interface = load_spec(REPO / 'meta_modelers_guide/composites/fig04b-cellular-interface.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig04b_cellular_interface)

In [ ]:
# === Edit parameters for composite 'The Cellular Interface' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'cell'  (local:CellularInterface)
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['interval'] = 1.0
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['summary'] = 'The cell as a bounded interface — senses environment, exposes exchange + goal ports'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['status'] = 'draft — typed interface; dynamics supplied by an executable handler'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['summary'] = 'The cell as a bounded interface — senses environment, exposes exchange + goal ports'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['description'] = 'The minimal cellular interface: a bounded, goal-directed cell that reads its environmental drivers and exposes physical exchange ports (chemical, mechanical, electrical, thermal) plus higher-level cellular ports (growth rate, shape, signaling, objective, viability). Chemical/mechanical exchange refine into subports. All ports carry biological units.'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['math'] = ['\\text{chemical} = -\\,k_{\\text{up}}\\,[\\text{chemical}_{\\text{ext}}]', '\\text{growth\\_rate} = \\mu_{\\max}\\,\\frac{[\\text{chemical}_{\\text{ext}}]}{K_m + [\\text{chemical}_{\\text{ext}}]}', '\\frac{d\\,\\text{viability}}{dt} = -A\\,e^{-E_a/RT}\\,\\text{viability}']
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['symbols']['k_up'] = 'uptake rate constant'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['symbols']['μ_max'] = 'maximum specific growth rate (hr⁻¹)'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['symbols']['K_m'] = 'half-saturation constant (mol·L⁻¹)'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['symbols']['A, E_a'] = 'Arrhenius prefactor & activation energy (E_a ≈ 300 kJ·mol⁻¹)'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['symbols']['T'] = 'absolute temperature (K)'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['ports']['chemical_ext'] = 'external nutrient concentration  (mol·L⁻¹)'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['ports']['mechanical_ext'] = 'external force / stress  (N)'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['ports']['electrical_ext'] = 'membrane driving potential  (mV)'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['ports']['thermal_ext'] = 'environmental temperature  (°C)'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['ports']['chemical'] = 'net nutrient uptake flux  (mol·s⁻¹)'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['ports']['mechanical'] = 'force exerted / felt  (N)'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['ports']['electrical'] = 'transmembrane current  (A)'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['ports']['thermal'] = 'heat flux  (W)'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['ports']['growth_rate'] = 'specific growth rate  (hr⁻¹)'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['ports']['shape'] = 'cell volume  (µm³)'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['ports']['signaling'] = 'signaling activity  (bits·s⁻¹)'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['ports']['objective'] = 'fitness objective  (dimensionless)'
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['contract']['ports']['viability'] = 'survival probability  (0–1)'

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-04 ===
STUDY = 'fig-04'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Figure 3**


In [ ]:
# Figure 3
show_viz(_render_one('svg:visualizations/figure_3.svg', {'caption': 'The minimal cellular interface: a cell exposes physical exchange ports (chemical, mechanical, electrical, thermal) and higher-level cellular ports (growth rate, shape, signaling, objective, viability).'}, RUNS_DB, STUDY_YAML))

## Study: Cell–environment coupling (`fig-05`)

**Question.** How does a cell exchange matter and signals with its environment?

**Purpose.** A cell couples to a spatio-flux nutrient field through one typed process — sensing, metabolizing, growing, and secreting.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `fig05-cell-environment` | `meta_modelers_guide.composites.fig05-cell-environment` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig05-cell-environment`** — `spec_meta_modelers_guide_composites_fig05_cell_environment` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig05_cell_environment = load_spec(REPO / 'meta_modelers_guide/composites/fig05-cell-environment.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig05_cell_environment)

In [ ]:
# === Edit parameters for composite 'Cell–Environment Coupling' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'reaction_diffusion'  (local:ReactionDiffusion)
spec_meta_modelers_guide_composites_fig05_cell_environment['state']['reaction_diffusion']['config']['interval'] = 1.0

# process 'production_degradation'  (local:ProductionDegradation)
spec_meta_modelers_guide_composites_fig05_cell_environment['state']['production_degradation']['config']['interval'] = 1.0

# process 'mechanical_stress'  (local:MechanicalStress)
spec_meta_modelers_guide_composites_fig05_cell_environment['state']['mechanical_stress']['config']['interval'] = 1.0

# process 'single_cell_processes'  (local:SingleCellProcesses)
spec_meta_modelers_guide_composites_fig05_cell_environment['state']['single_cell_processes']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-05 ===
STUDY = 'fig-05'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Figure 4**


In [ ]:
# Figure 4
show_viz(_render_one('svg:visualizations/figure_4.svg', {'caption': 'A cell couples to a spatio-flux nutrient field through one typed process — sensing, metabolizing, growing, and secreting.'}, RUNS_DB, STUDY_YAML))

## Study: Disintegration (`fig-06`)

**Question.** How is a process swapped between grains on the viability function?

**Purpose.** Loss of the cellular unit: a coarse-grained metabolism refines into its molecular mechanism, and the cell disintegrates into shed material.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `fig06b-grain-swap` | `meta_modelers_guide.composites.fig06b-grain-swap` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig06b-grain-swap`** — `spec_meta_modelers_guide_composites_fig06b_grain_swap` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig06b_grain_swap = load_spec(REPO / 'meta_modelers_guide/composites/fig06b-grain-swap.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig06b_grain_swap)

In [ ]:
# === Edit parameters for composite 'Grain Swap' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'grain_selector'  (local:GrainSelector)
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['grain_selector']['config']['interval'] = 1.0
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['grain_selector']['config']['summary'] = 'Grain selector — swap process grain on the viability function'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['grain_selector']['config']['contract']['status'] = 'draft — swap policy contract; no update dynamics yet'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['grain_selector']['config']['contract']['summary'] = 'Grain selector — swap process grain on the viability function'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['grain_selector']['config']['contract']['description'] = 'A generic multiscale controller. Two processes realize the SAME interface at different grains — a coarse, cheap description and a fine, mechanistic one. The selector watches the viability signal and swaps which grain is active: while the system is comfortably viable the coarse process suffices; as viability falls toward a critical boundary the fine-grained process is swapped in to resolve the mechanism that matters there, and lumped back out once viability recovers. Instantiated in this investigation as CPM (coarse) ⇄ particle dynamics (fine), but the contract is grain-agnostic.'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['grain_selector']['config']['contract']['math'] = ['\\text{active} = \\text{coarse} \\quad\\text{if}\\quad v \\geq \\theta_{\\uparrow}', '\\text{active} = \\text{fine} \\quad\\text{if}\\quad v \\leq \\theta_{\\downarrow}\\quad(\\theta_{\\downarrow}<\\theta_{\\uparrow}\\ \\text{: hysteresis})', '\\langle I_{\\text{coarse}}\\rangle = \\langle I_{\\text{fine}}\\rangle \\quad\\text{(both realize the same interface }I)']
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['grain_selector']['config']['contract']['symbols']['v'] = 'viability — survival margin (0–1)'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['grain_selector']['config']['contract']['symbols']['θ↑'] = 'swap-to-coarse threshold (recovered)'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['grain_selector']['config']['contract']['symbols']['θ↓'] = 'swap-to-fine threshold (stressed)'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['grain_selector']['config']['contract']['symbols']['I'] = 'shared external interface'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['grain_selector']['config']['contract']['ports']['viability'] = 'survival margin driving the swap  (0–1)'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['grain_selector']['config']['contract']['ports']['active_grain'] = 'selected grain — coarse | fine'

# process 'coarse_process'  (local:CoarseGrain)
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['coarse_process']['config']['interval'] = 1.0
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['coarse_process']['config']['summary'] = 'Coarse-grained process — cheap, lumped realization of the interface'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['coarse_process']['config']['contract']['status'] = 'draft — realizes the shared interface at one grain'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['coarse_process']['config']['contract']['summary'] = 'Coarse-grained process — cheap, lumped realization of the interface'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['coarse_process']['config']['contract']['description'] = "The low-resolution grain: a few aggregate state variables reproduce the interface's averaged behavior at low cost. Active while viability is high."
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['coarse_process']['config']['contract']['ports']['inflow'] = 'aggregate input flux  (mol·s⁻¹)'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['coarse_process']['config']['contract']['ports']['active_grain'] = "gate — runs when 'coarse' is selected"
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['coarse_process']['config']['contract']['ports']['biomass'] = 'aggregate biomass  (g)'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['coarse_process']['config']['contract']['ports']['energy'] = 'free energy  (J)'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['coarse_process']['config']['contract']['ports']['secretions'] = 'aggregate output flux  (mol·s⁻¹)'

# process 'fine_process'  (local:FineGrain)
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['fine_process']['config']['interval'] = 1.0
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['fine_process']['config']['summary'] = 'Fine-grained process — mechanistic, resolved realization of the interface'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['fine_process']['config']['contract']['status'] = 'draft — realizes the shared interface at one grain'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['fine_process']['config']['contract']['summary'] = 'Fine-grained process — mechanistic, resolved realization of the interface'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['fine_process']['config']['contract']['description'] = 'The high-resolution grain: the mechanism resolved into its constituent particles/reactions. Swapped in near a viability boundary, where the coarse average is no longer faithful.'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['fine_process']['config']['contract']['ports']['inflow'] = 'resolved input flux  (mol·s⁻¹)'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['fine_process']['config']['contract']['ports']['active_grain'] = "gate — runs when 'fine' is selected"
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['fine_process']['config']['contract']['ports']['biomass'] = 'biomass from resolved dynamics  (g)'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['fine_process']['config']['contract']['ports']['energy'] = 'free energy  (J)'
spec_meta_modelers_guide_composites_fig06b_grain_swap['state']['fine_process']['config']['contract']['ports']['secretions'] = 'resolved output flux  (mol·s⁻¹)'

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-06 ===
STUDY = 'fig-06'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Figure 5**


In [ ]:
# Figure 5
show_viz(_render_one('svg:visualizations/figure_5.svg', {'caption': 'Loss of the cellular unit: a coarse-grained metabolism refines into its molecular mechanism, and the cell disintegrates into shed material.'}, RUNS_DB, STUDY_YAML))

## Study: The molecular interface (`fig-07`)

**Question.** How is a molecular mechanism a process with typed physical channels?

**Purpose.** A molecular mechanism exposes physical ports (chemical, electrical, mechanical, thermal, structural) with molecular subports.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `fig07-molecular-mechanism` | `meta_modelers_guide.composites.fig07-molecular-mechanism` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig07-molecular-mechanism`** — `spec_meta_modelers_guide_composites_fig07_molecular_mechanism` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig07_molecular_mechanism = load_spec(REPO / 'meta_modelers_guide/composites/fig07-molecular-mechanism.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig07_molecular_mechanism)

In [ ]:
# === Edit parameters for composite 'A Molecular Mechanism' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'molecular_mechanism'  (local:MolecularMechanism)
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['interval'] = 1.0
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['summary'] = 'F₁Fₒ ATP synthase — a four-channel molecular transducer'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['status'] = 'draft — typed physical channels + behavior contract; no update dynamics yet'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['summary'] = 'F₁Fₒ ATP synthase — a four-channel molecular transducer'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['description'] = 'A molecular mechanism as a process with typed physical channels, defined over molecular structure (PDB/SMILES) with no cell-level boundary. The F₁Fₒ ATP synthase couples a transmembrane proton flux to rotary torque and ATP synthesis, conserving matter, charge, momentum, and energy across its four channels. The chemical channel refines into substrates, cofactors, catalysts, and products.'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['math'] = ['J_{\\text{H}^+} = g\\,\\Delta p = g\\,(\\Delta\\psi - \\tfrac{RT}{F}\\ln 10\\,\\Delta\\text{pH})', '\\tau = \\frac{n\\,F\\,\\Delta p}{2\\pi},\\qquad \\text{ATP} = J_{\\text{H}^+}/(n_{\\text{H}^+/\\text{ATP}})', '\\dot Q = J_{\\text{H}^+}\\,F\\,\\Delta p - \\tau\\,\\omega - \\Delta G_{\\text{ATP}}\\,\\text{ATP}']
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['symbols']['Δp'] = 'proton-motive force (mV)'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['symbols']['Δψ, ΔpH'] = 'membrane potential & pH gradient'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['symbols']['g'] = 'proton conductance'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['symbols']['τ'] = 'rotary torque (N·m)'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['symbols']['n'] = 'protons per turn'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['symbols']['F'] = 'Faraday constant'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['ports']['chemical_in'] = 'proton / substrate inflow  (mol·s⁻¹)'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['ports']['electrical_in'] = 'transmembrane proton current  (A)'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['ports']['mechanical_in'] = 'applied rotary load / torque  (N·m)'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['ports']['thermal_in'] = 'heat inflow  (W)'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['ports']['structure'] = 'molecular structure  (PDB / SMILES)'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['ports']['chemical_out'] = 'ATP synthesis flux  (mol·s⁻¹)'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['ports']['electrical_out'] = 'net charge translocated  (A)'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['ports']['mechanical_out'] = 'γ-shaft torque delivered  (N·m)'
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['contract']['ports']['thermal_out'] = 'dissipated heat  (W)'

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-07 ===
STUDY = 'fig-07'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Figure 6**


In [ ]:
# Figure 6
show_viz(_render_one('svg:visualizations/figure_6.svg', {'caption': 'A molecular mechanism exposes physical ports (chemical, electrical, mechanical, thermal, structural) with molecular subports.'}, RUNS_DB, STUDY_YAML))

## Study: Nested cellular hierarchy (`fig-08`)

**Question.** How do nested composites build a structural hierarchy?

**Purpose.** The cell's structural composition as a place graph — extracellular matrix, membrane, cytoplasm, nucleus, organelles — with the processes that act across levels.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `fig08-nested-hierarchy` | `meta_modelers_guide.composites.fig08-nested-hierarchy` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig08-nested-hierarchy`** — `spec_meta_modelers_guide_composites_fig08_nested_hierarchy` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig08_nested_hierarchy = load_spec(REPO / 'meta_modelers_guide/composites/fig08-nested-hierarchy.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig08_nested_hierarchy)

In [ ]:
# === Edit parameters for composite 'Nested Molecular Hierarchy' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'transmembrane_transport'  (local:TransmembraneTransport)
spec_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['transmembrane_transport']['config']['interval'] = 1.0

# process 'replication_and_repair'  (local:ReplicationAndRepair)
spec_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['replication_and_repair']['config']['interval'] = 1.0

# process 'cell_metabolism'  (local:CellMetabolism)
spec_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['cell_metabolism']['config']['interval'] = 1.0

# process 'transcription'  (local:Transcription)
spec_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['transcription']['config']['interval'] = 1.0

# process 'translation'  (local:Translation)
spec_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['translation']['config']['interval'] = 1.0

# process 'subunit_assembly'  (local:SubunitAssembly)
spec_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['subunit_assembly']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-08 ===
STUDY = 'fig-08'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Figure 7**


In [ ]:
# Figure 7
show_viz(_render_one('svg:visualizations/figure_7.svg', {'caption': "The cell's structural composition as a place graph — extracellular matrix, membrane, cytoplasm, nucleus, organelles — with the processes that act across levels."}, RUNS_DB, STUDY_YAML))

## Study: Self-organized processes (`fig-09`)

**Question.** How does a minimal cell coarse-grain into a self-organized process?

**Purpose.** Autopoiesis and the minimal cell: coarse-grained containment/metabolism/replication refine into molecular self-organized mechanisms.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `fig09b-minimal-cell` | `meta_modelers_guide.composites.fig09b-minimal-cell` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig09b-minimal-cell`** — `spec_meta_modelers_guide_composites_fig09b_minimal_cell` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig09b_minimal_cell = load_spec(REPO / 'meta_modelers_guide/composites/fig09b-minimal-cell.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig09b_minimal_cell)

In [ ]:
# === Edit parameters for composite 'The Minimal Cell' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'minimal_cell_containment'  (local:MinimalCellContainment)
spec_meta_modelers_guide_composites_fig09b_minimal_cell['state']['minimal_cell_containment']['config']['interval'] = 1.0

# process 'minimal_cell_metabolism'  (local:MinimalCellMetabolism)
spec_meta_modelers_guide_composites_fig09b_minimal_cell['state']['minimal_cell_metabolism']['config']['interval'] = 1.0

# process 'gene_expression'  (local:GeneExpression)
spec_meta_modelers_guide_composites_fig09b_minimal_cell['state']['gene_expression']['config']['interval'] = 1.0

# process 'minimal_cell_replication'  (local:MinimalCellReplication)
spec_meta_modelers_guide_composites_fig09b_minimal_cell['state']['minimal_cell_replication']['config']['interval'] = 1.0

# process 'diffusion'  (local:Diffusion)
spec_meta_modelers_guide_composites_fig09b_minimal_cell['state']['diffusion']['config']['interval'] = 1.0

# process 'reactions'  (local:Reactions)
spec_meta_modelers_guide_composites_fig09b_minimal_cell['state']['reactions']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-09 ===
STUDY = 'fig-09'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Figure 8**


In [ ]:
# Figure 8
show_viz(_render_one('svg:visualizations/figure_8.svg', {'caption': 'Autopoiesis and the minimal cell: coarse-grained containment/metabolism/replication refine into molecular self-organized mechanisms.'}, RUNS_DB, STUDY_YAML))

## Study: Growth & division (`fig-10-1`)

**Question.** How is cell division a place-graph rewrite?

**Purpose.** Cell growth and division as a place-graph rewrite: chromosome replication and segregation, then division into two daughter cells.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `fig10-1-division` | `meta_modelers_guide.composites.fig10-1-division` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig10-1-division`** — `spec_meta_modelers_guide_composites_fig10_1_division` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig10_1_division = load_spec(REPO / 'meta_modelers_guide/composites/fig10-1-division.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig10_1_division)

In [ ]:
# === Edit parameters for composite 'Cell Division — Draft Interface' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'dna_replication'  (local:DNAReplication)
spec_meta_modelers_guide_composites_fig10_1_division['state']['dna_replication']['config']['interval'] = 1.0

# process 'segregate_chromosome'  (local:SegregateChromosome)
spec_meta_modelers_guide_composites_fig10_1_division['state']['segregate_chromosome']['config']['interval'] = 1.0

# process 'divide'  (local:Divide)
spec_meta_modelers_guide_composites_fig10_1_division['state']['divide']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-10-1 ===
STUDY = 'fig-10-1'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Figure 9**


In [ ]:
# Figure 9
show_viz(_render_one('svg:visualizations/figure_9.svg', {'caption': 'Cell growth and division as a place-graph rewrite: chromosome replication and segregation, then division into two daughter cells.'}, RUNS_DB, STUDY_YAML))

## Study: Development (`fig-10-2`)

**Question.** How is development a place-graph rewrite?

**Purpose.** Multicellular development as a place-graph rewrite: a founder cell colonizes a surface and the biofilm accretes sibling cells.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `fig10-2-development` | `meta_modelers_guide.composites.fig10-2-development` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig10-2-development`** — `spec_meta_modelers_guide_composites_fig10_2_development` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig10_2_development = load_spec(REPO / 'meta_modelers_guide/composites/fig10-2-development.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig10_2_development)

In [ ]:
# === Edit parameters for composite 'Biofilm Development — Draft Interface' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'surface_attachment'  (local:SurfaceAttachment)
spec_meta_modelers_guide_composites_fig10_2_development['state']['surface_attachment']['config']['interval'] = 1.0

# process 'ecm_secretion'  (local:ECMSecretion)
spec_meta_modelers_guide_composites_fig10_2_development['state']['ecm_secretion']['config']['interval'] = 1.0

# process 'biofilm_growth'  (local:BiofilmGrowth)
spec_meta_modelers_guide_composites_fig10_2_development['state']['biofilm_growth']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-10-2 ===
STUDY = 'fig-10-2'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Figure 10**


In [ ]:
# Figure 10
show_viz(_render_one('svg:visualizations/figure_10.svg', {'caption': 'Multicellular development as a place-graph rewrite: a founder cell colonizes a surface and the biofilm accretes sibling cells.'}, RUNS_DB, STUDY_YAML))

## Study: Evolution (`fig-10-3`)

**Question.** How is evolution a place-graph rewrite?

**Purpose.** Evolution as a place-graph rewrite: a population establishes and diversifies into lineages under selection.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `fig10-3-evolution` | `meta_modelers_guide.composites.fig10-3-evolution` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig10-3-evolution`** — `spec_meta_modelers_guide_composites_fig10_3_evolution` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig10_3_evolution = load_spec(REPO / 'meta_modelers_guide/composites/fig10-3-evolution.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig10_3_evolution)

In [ ]:
# === Edit parameters for composite 'Evolution — Draft Interface' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'variation'  (local:Variation)
spec_meta_modelers_guide_composites_fig10_3_evolution['state']['variation']['config']['interval'] = 1.0

# process 'selection'  (local:Selection)
spec_meta_modelers_guide_composites_fig10_3_evolution['state']['selection']['config']['interval'] = 1.0

# process 'port_addition'  (local:PortAddition)
spec_meta_modelers_guide_composites_fig10_3_evolution['state']['port_addition']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: fig-10-3 ===
STUDY = 'fig-10-3'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Figure 11**


In [ ]:
# Figure 11
show_viz(_render_one('svg:visualizations/figure_11.svg', {'caption': 'Evolution as a place-graph rewrite: a population establishes and diversifies into lineages under selection.'}, RUNS_DB, STUDY_YAML))